# CME Futures: Model Analysis

This notebook reads the complete canonical model population produced by `06_linear` through
`10b_stochastic_discount_factor`. Each row retains family, configuration, label, checkpoint, fold
contract, training identity, and prediction identity. No null label is assigned to another
horizon, and checkpoints are not collapsed into a single model label.

IC measures whether a configuration ranks the cross-section correctly on a decision date. It
selects nothing: every row here proceeds to the equal-weight validation backtest in
`13_backtest`, where Sharpe performs selection and the checkpoint is part of what is selected.

What `ic_mean` and `ic_t` are, precisely, because the two readings are easy to confuse. Both are
computed **across folds**: `ic_mean` averages each fold's cross-sectional IC, and `ic_t` divides
that by the *standard error* of the same five numbers, which is their dispersion divided by the
square root of how many are defined - not the dispersion itself. `ic_std` in the table below is
the dispersion, so reproducing `ic_t` from the two columns needs the `sqrt(n_folds_ic)` factor.

The registry also computes the daily-series reading with its HAC standard error, which is the
inferential statistic, but the predictions reader does not surface it, so it is not in the table
below. `ic_n_days` is carried instead: it
counts the validation dates that produced a defined IC, and a configuration whose predictions
collapse to near-constant on some dates has its `ic_mean` measured over fewer of them. Reading
`ic_mean` without `ic_n_days` is how a partial-coverage artifact reads as a leader.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：模型分析

本 Notebook 讀取 `06_linear` 至 `10b_stochastic_discount_factor` 產生的完整正式模型群組。每列均保留模型類別、設定、標籤、檢查點、資料折規約、訓練識別資訊及預測識別資訊。空白標籤不會被指派給另一個預測期，檢查點也不會合併成單一模型標籤。

資訊係數衡量某項設定在決策日是否正確排列商品的橫斷面。它不負責選擇：此處每列都會進入 `13_backtest` 的等權重驗證回測，由夏普比率決定選擇，檢查點也是選擇對象的一部分。

須精確區分 `ic_mean` 與 `ic_t`，因為兩者容易混淆。兩者都**跨資料折**計算：`ic_mean` 將各折的橫斷面資訊係數取平均；`ic_t` 再除以相同五個數值的*標準誤*，也就是其離散程度除以已定義數值個數的平方根，而非直接除以離散程度。下表的 `ic_std` 是離散程度，因此若要由這兩欄重現 `ic_t`，須納入 `sqrt(n_folds_ic)` 這項因子。

登錄表也會以每日序列及其 HAC 標準誤計算另一種解讀；那是用於推論的統計量，但預測讀取器未將其顯示，因此不在下表。此處改為保留 `ic_n_days`：它計算資訊係數有定義的驗證日期數。如果某項設定的預測在部分日期幾乎變成常數，其 `ic_mean` 便只根據較少日期計算。只看 `ic_mean`、不看 `ic_n_days`，就可能將涵蓋不完整的產出誤認為領先者。
<!-- ML4T_ZH_END c0000-p000 -->


## What an information coefficient measures, and what it cannot

Everything in the table below is built on the IC, so it is worth being exact about what the
number is before reading any of it.

On one decision date there is a set of products, a predicted return for each, and the return
each actually went on to earn. The IC is the rank correlation between those two lists. It asks
a deliberately narrow question: did the model put them **in the right order**? It does not ask
whether the predicted magnitudes were close, and it cannot - a model that predicts every return
at a hundredth of its true size scores exactly as well as one that gets the levels right, as
long as the ordering matches.

That narrowness is the point for a strategy of this shape. The backtest in `13_backtest` holds
the top-ranked products and shorts the bottom-ranked ones in equal weight, so the ordering is
the entire input and the magnitudes are discarded before a position is taken. A diagnostic that
rewarded accurate levels would be measuring something the strategy never uses.

**It is computed per decision date and then averaged, never pooled.** Pooling every
product-date into one correlation would let a period when the whole market moved together
masquerade as skill at telling products apart: on a day when everything rallies, a model that
ranks products at random still shows agreement between its predictions and the outcomes if the
cross-sections are stacked. Correlating within a date and averaging afterwards removes the
common move by construction, because it is the same for every product on that date.

### Why a good IC is a small number


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 資訊係數衡量什麼，以及無法衡量什麼

下表所有內容都建立在資訊係數上，因此在閱讀表格前，應先精確理解這個數值。

在某個決策日，有一組商品、每項商品的預測報酬，以及它日後實際取得的報酬。資訊係數是這兩份清單的排名相關性。它刻意只問一個範圍狹窄的問題：模型是否把商品排成**正確順序**？它不問預測幅度是否接近實際值，也無法衡量這點。只要排序相同，將所有報酬預測為真實值百分之一的模型，與準確預測數值水準的模型，得分完全相同。

對這種形式的策略而言，範圍狹窄正是目的。`13_backtest` 的回測等權重持有排名最高的商品、放空排名最低的商品，因此排序是全部輸入；在建立部位前，預測幅度便已被捨棄。若診斷指標獎勵數值水準準確，就會衡量策略根本未使用的資訊。

**資訊係數逐決策日計算，再取平均，絕不將資料混合後一次計算。**若把所有商品與日期堆在一起算單一相關性，整體市場同向變動的時期便可能偽裝成模型區分商品的能力：某日若所有商品都上漲，即使模型隨機排序商品，把各日橫斷面堆疊後，預測與結果仍可能顯示一致。先逐日計算相關性再取平均，在建構上就排除了共同變動，因為同一天的共同變動對每項商品都相同。

### 為何好的資訊係數仍是小數值
<!-- ML4T_ZH_END c0001-p000 -->

A reader arriving from a forecasting background should expect these to look disappointing. A
monthly-horizon equity or futures IC of 0.03 to 0.05 is a real, usable signal; 0.10 sustained
would be remarkable. This is not a weak result being excused - it is what predicting an
overwhelmingly noise-dominated quantity looks like when it works. The edge comes from applying
a small consistent tilt across many products and many dates, not from being right about any one
of them, and the arithmetic of that is what `13_backtest` measures and this notebook does not.

### Why the ranking diagnostic selects nothing

A high IC does not imply a tradeable strategy, and this is the boundary the notebook's title
refers to. A configuration can rank the cross-section well and still lose money: if its ranking
churns from one decision to the next, the turnover it implies costs more than the tilt earns; if
its skill sits entirely in the products that are least liquid, the positions cannot be taken at
the prices the backtest assumes. Neither of those is visible in an IC, because an IC has no
notion of holding anything.

So nothing here is a decision. Every complete row in this catalog proceeds to the equal-weight
validation backtest, where Sharpe selects and the checkpoint is part of what is selected. This
notebook is where a reader forms an expectation and, more usefully, notices where the backtest
later disagrees with it - a configuration that ranks well and backtests badly is the most
informative row in the whole case study, because the gap between the two is exactly where
turnover and tradeability live.

<!-- ML4T_ZH_BEGIN c0001-p001 -->
熟悉一般預測問題的讀者，可能會覺得這些數值令人失望。股票或期貨在一個月預測期的資訊係數若為 0.03 至 0.05，已是真實且可用的訊號；若能持續達到 0.10，則相當出色。這不是替薄弱結果找藉口，而是預測一個幾乎完全由雜訊主導的量，且預測確實有效時會出現的樣貌。優勢來自對多項商品及多個日期持續施加小幅傾斜，而非在某單項商品上猜對；其計算由 `13_backtest` 衡量，不是本 Notebook 的工作。

### 為何排名診斷不負責選擇

資訊係數高，不代表策略可交易；這正是 Notebook 標題所指的界線。某項設定可能很會排列商品，卻仍虧錢：若其排名每次決策都大幅變動，隱含的換手成本可能超過傾斜部位的收益；若預測能力全落在流動性最差的商品上，就無法以回測假設的價格建立部位。資訊係數完全不涉及持有部位，因此看不出這些問題。

所以此處不做決策。目錄中每列完整資料都會進入等權重驗證回測，由夏普比率選擇，檢查點也屬於選擇對象。本 Notebook 可讓讀者先形成預期，更重要的是辨認回測日後在哪裡與預期不同：排名良好但回測不佳的設定，是整個案例研究中最有資訊價值的資料列，因為兩者差距正是換手與可交易性發揮影響之處。
<!-- ML4T_ZH_END c0001-p001 -->


In [1]:
"""Analyze complete CME futures model and causal result catalogs."""

import json

import numpy as np
import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    CASE_STUDY,
    MODEL_POPULATION_NAMES,
    official_prediction_catalog,
    open_study,
    product_universe_table,
)
from case_studies.research import CausalResult, require_declared_menu_coverage
from utils.paths import get_case_study_dir

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None

## Complete prediction catalog

The six official population snapshots were created before their model runs. Opening all six and
calling `require_complete` means a failed configuration or checkpoint cannot disappear from this
analysis because another row happened to finish.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 完整預測目錄

六個正式模型群組快照都在各自模型執行前建立。開啟全部六份並呼叫 `require_complete`，可確保某項設定或檢查點即使失敗，也不會因另一資料列恰好完成，就從此分析中消失。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
if EXECUTION_TIER == "preview" and WORKSPACE is None:
    raise ValueError("preview execution requires WORKSPACE")
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


Canonical analysis reads the six published population snapshots, which is the whole point of
freezing them before their runs. A preview has no published population to read: it is a reduced
re-execution whose rows exist only in its own workspace and which is deliberately excluded from
every official population. It reads its own complete validation predictions instead, and the
comparison against the declared menus below is skipped with them, because a preview fits a named
subset by design and would fail that comparison on every configuration it left out.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
正式分析讀取六個已發布模型群組快照；在執行前先將其凍結，目的正是如此。預覽沒有已發布的模型群組可讀：它是縮減範圍後重新執行，資料列只存在自己的工作區，並刻意排除於所有正式群組之外。預覽改讀自己完整的驗證預測；下方與已宣告選單的比較也會跳過，因為預覽依設計只擬合部分具名設定，對每項未納入的設定都會無法通過完整性比較。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
if EXECUTION_TIER == "canonical":
    catalog = official_prediction_catalog(study, MODEL_POPULATION_NAMES)
else:
    catalog = (
        study.predictions.table(include_preview=True)
        .filter(
            (pl.col("execution_tier") == "preview")
            & (pl.col("split") == "validation")
            & pl.col("complete")
        )
        .sort("label", "family", "config_name", "checkpoint_kind", "checkpoint_value")
    )
    if catalog.is_empty():
        raise RuntimeError("preview execution registered no complete validation predictions")


def _feature_count(spec_json: str) -> int:
    spec = json.loads(spec_json)
    computation = spec.get("computation", spec)
    return len(computation.get("feature_names") or [])


analysis = catalog.with_columns(
    pl.col("spec_json").map_elements(_feature_count, return_dtype=pl.Int64).alias("feature_count")
).select(
    "family",
    "config_name",
    "label",
    "checkpoint_kind",
    "checkpoint_value",
    "feature_count",
    "ic_mean",
    "ic_t",
    "ic_n_days",
    "n_folds",
    "training_hash",
    "prediction_hash",
)

### Every declared model is here

Each execution notebook checks that it produced everything **it** requested, so none of them can
see a configuration that no notebook requests at all - a menu entry nobody claimed publishes
nothing and every completeness check still passes. This is the one place the families reassemble,
so it is the only place that check can be made. `require_declared_menu_coverage` compares
`(family, label, config_name)` against the training menus and raises on either direction: a
declared model the population omits, or a model in the population that no menu declares.

It returns the rows knowingly excluded, so what this notebook is missing is displayed rather than
taken on trust. `causal_dml` is not in the comparison - it is not a predictive family and the
adapter registry, not a list here, is what decides that.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
### 每個已宣告模型都在此處

每份執行用 Notebook 都會檢查是否產出它**自己**提出的全部請求，因此無法發現完全沒有任何 Notebook 提出請求的設定：選單中某項設定若沒有人認領，就不會發布任何內容，但每項完整性檢查仍會通過。各模型類別只在此處重新合併，因此也只有這裡能執行該檢查。`require_declared_menu_coverage` 將 `(family, label, config_name)` 與訓練選單比較，任一方向不符都會引發錯誤：已宣告卻不在模型群組內，或群組內有模型卻沒有選單宣告。

它也會傳回明確排除的資料列，讓本 Notebook 缺少哪些內容直接顯示，而非只憑信任。`causal_dml` 不在比較範圍內，因為它不是預測模型類別；判定這點的是轉接器登錄表，而非此處的清單。
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
if EXECUTION_TIER == "canonical":
    excluded = require_declared_menu_coverage(analysis, case_study=CASE_STUDY)
else:
    excluded = analysis.clear()
excluded

family,label,config_name,reason
str,str,str,str


In [6]:
analysis.sort("label", "family", "config_name", "checkpoint_value")

family,config_name,label,checkpoint_kind,checkpoint_value,feature_count,ic_mean,ic_t,ic_n_days,n_folds,training_hash,prediction_hash
str,str,str,str,i64,i64,f64,f64,f64,f64,str,str
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",5,71,-0.033954,-0.941949,1269.0,5.0,"""79752926dbf0""","""31e6a5c64228"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",10,71,-0.036698,-0.962205,1269.0,5.0,"""79752926dbf0""","""243d0d3c7165"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",15,71,-0.039259,-0.927042,1269.0,5.0,"""79752926dbf0""","""2969c6f856d3"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",20,71,-0.035184,-0.959816,1269.0,5.0,"""79752926dbf0""","""74c22a14939b"""
"""deep_learning""","""lstm_h64""","""fwd_ret_21d""","""epoch""",25,71,-0.024894,-0.589233,1269.0,5.0,"""79752926dbf0""","""554aa3766fca"""
…,…,…,…,…,…,…,…,…,…,…,…
"""tabular_dl""","""tabm_s""","""fwd_ret_5d""","""epoch""",100,69,-0.018488,-1.9104,1285.0,5.0,"""711c4e1f5d53""","""c1a0ea8bc19a"""
"""tabular_dl""","""tabm_s""","""fwd_ret_5d""","""epoch""",125,69,-0.023885,-1.867816,1285.0,5.0,"""711c4e1f5d53""","""08daba18b64f"""
"""tabular_dl""","""tabm_s""","""fwd_ret_5d""","""epoch""",150,69,-0.022327,-1.690648,1285.0,5.0,"""711c4e1f5d53""","""3e69f61325d2"""


## Interpretation boundaries

The table compares ranking diagnostics under the declared walk-forward protocol. The backtest
engine supplies portfolio returns, transaction costs, contract sizing, and roll execution before
selection.

The distinction is worth stating as a rule rather than as a caveat: **every quantity in this
notebook is a property of the predictions, and every quantity that decides anything is a
property of a portfolio.** A prediction has no size, no holding period, no financing and no
execution price. Those enter in `13_backtest`, and they are capable of reordering the table
below completely - which is why a leaderboard here is a hypothesis about the backtest rather
than a preview of it.

Conformal weighting, when used later as an allocator, calibrates chronologically from prior
validation observations. It uses the calibration-window scale and the finite-sample higher order
statistic. It does not fit a scale on the evaluation fold or pool all folds before calibration.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 解讀界線

表格比較依已宣告滾動向前規約計算的排名診斷。回測引擎在選擇前，會提供投資組合報酬、交易成本、合約口數及展期執行結果。

這項差異應明定為規則，而不只是附帶提醒：**本 Notebook 的每個數量都是預測的性質，而所有真正決定選擇的數量都是投資組合的性質。**預測本身沒有部位大小、持有期間、融資或成交價格。這些會在 `13_backtest` 中加入，而且足以完全改變下表排名。因此，此處排行榜是對回測的假設，而非回測結果的預覽。

若日後以保形方法加權進行配置，校準會依時間順序使用先前的驗證觀測值，採用校準視窗的尺度及有限樣本的較高階次序統計量。不會在評估折上擬合尺度，也不會在校準前將所有資料折混合。
<!-- ML4T_ZH_END c0011-p000 -->


## Causal diagnostics

Double machine learning answers a different question from prediction. The treatment effect is
conditioned on the configured confounders, and HAC uncertainty follows the decision-time order.
A covariance-estimator failure is not relabeled as HAC. The shared runner must return a finite HAC
standard error for a result to be complete.

**The `refutation_p` column below is not evidence that these effects survived a placebo test.**
The refutation permutes contiguous blocks within each product, and the shared runner sizes those
blocks as `max(label_buffer, treatment_window)`. `causal.treatment_window` is 1 here, so the label
buffer binds and the registered rows carry a 21-period block for `fwd_ret_21d` and a 5-period
block for `fwd_ret_5d`. Neither length is a property of `carry_pct`, whose own persistence the
cell below measures on this case study's feature panel: the autocorrelation is pooled within
product, each product demeaned before pooling so a level difference between products cannot
stand in for persistence within one, and on one row per product-session, because the block
counts sessions.


<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 因果診斷

雙重機器學習回答的問題與預測不同。處置效果以已設定的混雜因子為條件，HAC 不確定性估計遵循決策時間順序。共變異數估計器若失敗，不會被重新標示成 HAC。共用執行器必須傳回有限值的 HAC 標準誤，結果才算完整。

**下方 `refutation_p` 欄，不能作為這些效果通過安慰劑檢驗的證據。**反證程序會在每項商品內置換連續區塊，共用執行器將區塊長度設為 `max(label_buffer, treatment_window)`。此處 `causal.treatment_window` 為 1，因此由標籤緩衝長度決定；登錄資料列對 `fwd_ret_21d` 採 21 期區塊，對 `fwd_ret_5d` 採 5 期區塊。兩種長度都不是 `carry_pct` 自身的性質。下方儲存格會從本案例研究的特徵面板，衡量其持續性：先對每項商品去平均，再彙整商品內的自相關，避免商品間水準差異被誤認為單一商品內的持續性；每項商品每個交易日取一列，因為區塊計算的是交易日數。
<!-- ML4T_ZH_END c0012-p000 -->

**The two blocks sit at very different points on that profile, so the concern bears much more
on one label than the other.** Read the block lengths against the autocorrelation at those
lags rather than against the half-life: the decay is slower than the AR(1) half-life implies -
an AR(1) with this lag-1 value would sit at 0.38 by lag 5 and 0.02 by lag 21, where the panel
is at 0.52 and 0.14 - so the half-life is a lower bound on persistence, not the yardstick for
the block. At the 5-session block used for `fwd_ret_5d` the autocorrelation is still 0.52, so that
block leaves real dependence unpreserved; the placebo is a weaker opponent than the truth and
`fwd_ret_5d`'s empirical p-value is biased toward zero by some amount this notebook does not
quantify. At the 21-session block used for `fwd_ret_21d` it is 0.14, and indistinguishable
from zero by lag 63, so that block spans most of the dependence and the concern is
correspondingly weaker there.


<!-- ML4T_ZH_BEGIN c0012-p001 -->
**兩種區塊在持續性曲線上的位置很不同，因此疑慮對其中一種標籤的影響遠大於另一種。**應以各區塊長度對照該落後期的自相關，而非以半衰期判斷：衰減比 AR(1) 半衰期暗示的更慢。若 AR(1) 具有此處的落後一期數值，落後 5 期時會是 0.38、落後 21 期時會是 0.02，但面板分別為 0.52 與 0.14。因此，半衰期是持續性的下界，而不是區塊長度的衡量標準。`fwd_ret_5d` 使用的 5 日區塊，落後 5 期的自相關仍為 0.52，故未保留真實相依性；安慰劑檢驗比真實情況更容易通過，`fwd_ret_5d` 的經驗 p 值會在本 Notebook 未量化的程度上偏向零。`fwd_ret_21d` 使用的 21 日區塊，該值為 0.14，至落後 63 期則已無法與零區分，因此 21 日區塊涵蓋大部分相依性，相關疑慮也較輕。
<!-- ML4T_ZH_END c0012-p001 -->

**That is no longer what the column reports, and the reason is worth following.** `fwd_ret_5d`
used to sit at 0.0396 and `fwd_ret_21d` at 0.0099, which is 1/101 and the floor 100 draws can
report. The refutation now compares HAC t-statistics rather than raw effects, because a permuted
treatment is not predictable from the controls, its residual keeps nearly all its variance, and
that variance is the denominator of the second-stage effect - so every placebo effect was divided
by a larger number than the observed one. Correcting that moved `fwd_ret_5d` to 0.5545 and
`fwd_ret_21d` to 0.2673, both `Fails`, on an identical fit. The block-length argument above is a
separate, uncorrected narrowing and it bears mainly on `fwd_ret_5d`; either way it is no longer
visible in these two numbers. Read the DML point estimate and its HAC standard error. The
refutation column is recorded for completeness and carries no evidence here.

<!-- ML4T_ZH_BEGIN c0012-p002 -->
**但該欄目前已不再報告當時的數值，原因值得追查。**`fwd_ret_5d` 原為 0.0396，`fwd_ret_21d` 原為 0.0099；後者是 1/101，也是抽樣 100 次能報告的最低值。反證程序現在比較 HAC t 統計量，而非原始效果，因為置換後的處置變數無法由控制變數預測，其殘差幾乎保留全部變異數；該變異數是第二階段效果估計的分母。因此，每個安慰劑效果原先都除以比實際觀察效果更大的數值。修正後，在相同擬合下，`fwd_ret_5d` 變為 0.5545，`fwd_ret_21d` 變為 0.2673，兩者皆為 `Fails`。上述區塊長度問題是另一項尚未修正的分布縮窄，主要影響 `fwd_ret_5d`；無論如何，它已無法從目前這兩個數值看出。應閱讀雙重機器學習的點估計與 HAC 標準誤。反證欄僅為完整記錄而保留，此處不具證據力。
<!-- ML4T_ZH_END c0012-p002 -->


In [7]:
# The panel carries one row per contract position, so a product-session appears up to three
# times with the same carry_pct. Lagging without de-duplicating steps ~2.78 rows per session
# and reports a persistence profile stretched by that factor. The block the refutation permutes
# counts sessions - run_dml_analysis requires strictly increasing timestamps within a product -
# so sessions are the scale the two have to be compared on.
carry = (
    pl.read_parquet(get_case_study_dir(CASE_STUDY) / "features" / "financial.parquet")
    .select(["product", "timestamp", "carry_pct"])
    .drop_nulls()
    .unique(subset=["product", "timestamp"])
    .sort(["product", "timestamp"])
)
autocorr = []
for lag in (1, 5, 21, 63):
    paired = (
        carry.with_columns(pl.col("carry_pct").shift(lag).over("product").alias("lagged"))
        .drop_nulls()
        .with_columns(
            (pl.col("carry_pct") - pl.col("carry_pct").mean().over("product")).alias("x"),
            (pl.col("lagged") - pl.col("lagged").mean().over("product")).alias("y"),
        )
    )
    rho = (paired["x"] * paired["y"]).sum() / (
        ((paired["x"] ** 2).sum() * (paired["y"] ** 2).sum()) ** 0.5
    )
    autocorr.append({"lag": lag, "autocorrelation": rho, "n_pairs": paired.height})
carry_persistence = pl.DataFrame(autocorr)
half_life = np.log(0.5) / np.log(carry_persistence["autocorrelation"][0])
print(
    f"carry_pct within-product pooled autocorrelation, "
    f"AR(1) half-life {half_life:.1f} sessions from lag 1"
)
carry_persistence

carry_pct within-product pooled autocorrelation, AR(1) half-life 3.6 sessions from lag 1


lag,autocorrelation,n_pairs
i64,f64,i64
1,0.824105,110782
5,0.52142,110662
21,0.141984,110182
63,-0.010403,108922


In [8]:
causal_rows = []
for label in ALL_LABELS:
    result = CausalResult.one(study, label=label, execution_tier=EXECUTION_TIER)
    if not result.complete:
        raise RuntimeError(f"causal result for {label} is incomplete")
    causal_rows.append({"label": label, "causal_hash": result.hash, **result.metrics})
causal = pl.DataFrame(causal_rows).sort("label")

In [9]:
causal

label,causal_hash,n_obs,dml_effect,dml_se_hac,p_value_hac,naive_effect,confounding_bias_pct,refutation_p,refutation_n_successful,placebo_effects,placebo_t_stats,placebo_frozen_fraction,refutation_class
str,str,i64,f64,f64,f64,f64,f64,f64,i64,list[f64],list[f64],f64,str
"""fwd_ret_21d""","""8d7cc0d7a423""",73239,-0.016256,0.008739,0.062963,-0.008765,46.084494,0.267327,100,"[0.000774, 0.005316, … 0.003168]","[0.358198, 2.078782, … 1.56593]",0.008819,"""Fails"""
"""fwd_ret_5d""","""a50a55ef32e8""",74057,-0.002298,0.002474,0.352955,-0.002069,9.983893,0.554455,100,"[0.001101, 0.002165, … 0.000665]","[1.465491, 2.992561, … 0.889154]",0.002854,"""Fails"""


## What proceeds to backtesting

All complete prediction rows proceed. The next notebook passes the selected Polars rows directly
to the shared backtest call, publishes product-keyed decisions, and records contract, roll, price,
and prediction lineage. Validation backtest Sharpe, with the prediction checkpoint included in the
configuration identity, is the selection statistic.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 哪些內容進入回測

所有完整的預測資料列都會進入下一階段。下一份 Notebook 將選定的 Polars 資料列直接傳給共用回測呼叫，發布以商品為鍵值的決策，並記錄合約、展期、價格及預測的來源紀錄。驗證回測夏普比率是選擇統計量，預測檢查點也包含在設定識別資訊中。
<!-- ML4T_ZH_END c0016-p000 -->
